# 02. Model Architecture Experiments & Benchmarks

This notebook evaluates model architectures, inference latency, and loss dynamics:
1. Loading CRNN / CNN-CTC models with edge configurations
2. Latency & memory benchmarking (FP32 vs FP16 on GPU/CPU)
3. Forward pass verification and CTC output shape inspection
4. Normalized Edit Distance (NED) calculation on synthetic test batches

In [ ]:
import time
import torch
import yaml
from pathlib import Path
from src.models.model import CRNNEdgeRecognizer
from src.utils.metrics import compute_normalized_edit_distance, compute_accuracy

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

# Instantiate model
model = CRNNEdgeRecognizer(in_channels=3, num_classes=52, hidden_size=128)
model.to(device)
model.eval()

# Benchmark inference latency
dummy_input = torch.randn(1, 3, 64, 192, device=device)
with torch.no_grad():
    # Warmup
    for _ in range(10):
        _ = model(dummy_input)
    
    start_time = time.perf_counter()
    runs = 100
    for _ in range(runs):
        output = model(dummy_input)
    avg_latency_ms = ((time.perf_counter() - start_time) / runs) * 1000

print(f"Output Shape: {output.shape} (TimeSteps x Batch x VocabSize)")
print(f"Average Latency: {avg_latency_ms:.2f} ms per frame")

In [ ]:
# Metric validation check
preds = ["1กข1234", "ขค5678", "2กด9999"]
targets = ["1กข1234", "ขค5670", "2กด9999"]

acc = compute_accuracy(preds, targets)
ned = compute_normalized_edit_distance(preds, targets)
print(f"Exact Match Accuracy: {acc * 100:.2f}%")
print(f"Mean Normalized Edit Distance: {ned:.4f}")